# Week 1: System exploration (Group)

In the first stage, your aim is to **understand the behaviour of a system**.

You should focus on questions like:
- How do the observed signals change over time?
- How do different input patterns affect the observations?
- Are some signals more informative than others?
- What is easy to observe, and what seems hidden?

At this stage, the priority is **exploration and intuition-building**, not solving the whole project.

## Group tasks for Week 1

As a group, extend the basic experiment and **build a shared code workflow.**

Your group should aim to produce:
- a clean way to explore dataset, either from simulation or experiment,
- standard plotting functions,
- a simulation platform for studying and comparing effects of different input patterns and system dynamics (transition matrices), for example:
    - random inputs,
    - pulses,
    - oscillatory inputs,
    - activating only one input channel at a time,
    - activating different channels with different amplitudes,
    - exploring the relationship between input structure and system dynamics (such as coupling between inputs and transition matrices);
- brief documentation so that all group members can build on the same starting point.

All of the above tasks should be completed by Friday 22 May, before the scheduled check-in session.

## Week 1 deliverable

**Deliverable:** Group simulation code + brief documentation  
**Marks:** 10 group marks  
**Due:** Friday 22 May, 11am–1pm (**compulsory session**)

At the check-in session, the demonstrator will review the code and ask each member of the group questions about it. Every member of the group is therefore expected to have contributed to the code and to take full responsibility for understanding it.

## Example dataset

In [1]:
# loading dataset
import numpy as np
data = np.load("ExampleDataset.npy")
print("Dataset loaded successfully. Shape:", data.shape)  # (Trials, Timepoints, Neurons) - so we batch 
print(data[0][50],data[4][50])

Dataset loaded successfully. Shape: (5, 60, 16)
[3.81957978 4.14909607 1.88203954 4.76866    3.32711863 4.27144729
 3.56429751 2.48635852 4.08573152 3.67330504 4.83260623 3.05891043
 4.35042048 2.88464824 4.55675246 4.6058064 ] [3.81957978 4.14909607 1.88203954 4.76866    3.32711863 4.27144729
 3.56429751 2.48635852 4.08573152 3.67330504 4.83260623 3.05891043
 4.35042048 2.88464824 4.55675246 4.6058064 ]


## Desgin an illustrator

A class that can generate primary statistics, analyses, and visualization of any dataset. Design your illustrator in **Illustrator.py**.

To test your documentation, the demonstrator will use your illustrator here, but will never see your code (in this section). 

Please make your annotations clear and comprehensive!

In [2]:
from Illustrator import Illustrator
from Explorer import Explorer
Illustrator = Illustrator(data)
    # Instantiate the class
explorer = Explorer(data)

# 2. Fit the continuous dynamics
explorer.fit_lds_em(state_dim=2, n_iter=10)

# 3. Detect the discrete operational phases (e.g., 3 modes)
explorer.fit_hmm(n_states=3)

# 4. Decode the external kinematics 
# (Assume 'force_data' is a numpy array of shape [Trials, Timepoints, 1])

    # Get the covariance matrix (returns a 20x20 numpy array)

# Launch the interactive GUI
explorer.interactive_dashboard()
pass

## Design a simulator

In GG4, we use a simple linear model:
$$
x_{t+1} = Ax_t + Bu_t + w_t, w_t \sim \mathcal{N}(0, Q)
$$
$$
y_t = Cx_t + o_t, o_t \sim \mathcal{N}(0, R)
$$

Your simulator should be able to simulate observation of any length given different parameters.

Design your simulator in **Simulator.py**.

To test your documentation, the demonstrator will use your simulator here, but will never see your code (in this section). 
Please make your annotations clear and comprehensive!

Combining the illustrator and simulator, what's your findings? Illustrate them.
This forms your answer to the question: "effects of different input patterns and system dynamics"

In [4]:
import Simulator as sim


In [9]:
import numpy as np
from IPython.display import clear_output  # <-- Add this line right here!
from Explorer import Explorer
from Simulator import Simulator
# 1. Load your real assignment dataset
# (Make sure "ExampleDataset.npy" is in the same folder as your notebook)
real_data = np.load("ExampleDataset.npy")

# 2. Define your control functions for the simulator
def step_controller(time, output):
    if time <= 10:
        return [1,1]
    return [0,0]

def sinusoidal_controller(time, states):
    return np.array([
        np.sin(time / 5),
        np.cos(time / 10)
    ])

controllers = {
    'Sinusoidal': sinusoidal_controller,
    'Step': step_controller
}

# 3. Define your initial guessed matrices for the Simulator
dt = 0.1
sim_matrices = [
    # A Matrix (Dynamics)
    [[0.98, 0, dt, 0], [0, 0.98, 0, dt], [0, 0, 0.98, 0], [0, 0, 0, 0.98]],
    # B Matrix (Control)
    [[0, 0], [0, 0], [1, 0], [0, 1]],
    # C Matrix (Observation)
    [[1, 0, 0, 0], [0, 1, 0, 0]],
    # Q Matrix (Process Noise)
    [[0.01, 0, 0, 0], [0, 0.01, 0, 0], [0, 0, 0.05, 0], [0, 0, 0, 0.05]],
    # R Matrix (Observation Noise)
    [[0.5, 0], [0, 0.5]]
]
sim = Simulator(sim_matrices)

# 4. Instantiate the Explorer with your REAL data as the baseline
explorer = Explorer(real_data)

# 5. Fit the LDS and HMM models to your real data right away 
# (This might take a few seconds)
print("Fitting models to real baseline data...")
explorer.fit_lds_em(state_dim=2, n_iter=10)
explorer.fit_hmm(n_states=3)
clear_output() # Clears the print statement to keep the notebook clean

# 6. Attach the simulator to the explorer
explorer.attach_simulator(sim, Simulator, controllers)

# 7. Launch the dashboard!
explorer.interactive_dashboard()